In [3]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from v2_eval import assign_block_folds, iter_block_splits

raw = pd.read_csv("../data/raw/resume_data_for_ranking.csv")
feat = pd.read_parquet("../data/processed/v2_features.parquet")
assert len(raw) == len(feat)

In [4]:
job_dependent_in_resume_block = ["responsibilities"]

resume_cols = [c for c in raw.columns[:28] if c not in job_dependent_in_resume_block]
job_cols = ["job_position_name", "educationaL_requirements", "experiencere_requirement",
            "age_requirement", "responsibilities.1", "skills_required"] + job_dependent_in_resume_block
assert raw.columns[28] == "job_position_name"

def row_key(df):
    return df.apply(lambda r: "|".join(map(str, r.values)), axis=1)

feat["resume_id"] = pd.factorize(row_key(raw[resume_cols]))[0]
feat["job_id"] = pd.factorize(row_key(raw[job_cols]))[0]

print("Distinct resumes:", feat["resume_id"].nunique(), "(expected ≈ 340)")
print("Distinct jobs:", feat["job_id"].nunique(), "(expected 28)")
print("Rows per resume :\n", feat.groupby("resume_id").size().describe().round(1))
print("Distinct values of experience_required_years per job (max):",
      feat.groupby("job_id")["experience_required_years"].nunique().max(), "(expected 1)")

Distinct resumes: 344 (expected ≈ 340)
Distinct jobs: 28 (expected 28)
Rows per resume :
 count    344.0
mean      27.7
std        2.2
min        4.0
25%       28.0
50%       28.0
75%       28.0
max       28.0
dtype: float64
Distinct values of experience_required_years per job (max): 1 (expected 1)


In [5]:
skills_ok = (feat.groupby("job_id")["skills_soft_coverage"]
             .apply(lambda s: int(s.notna().mean() > 0.5)).to_dict())
print("Jobs without required skills :", sum(v == 0 for v in skills_ok.values()))

feat["resume_fold"], feat["job_fold"] = assign_block_folds(
    feat["resume_id"], feat["job_id"],
    n_resume_folds=5, n_job_folds=5, seed=42, job_strata=skills_ok,
)

Jobs without required skills : 5


In [6]:
seen = np.zeros(len(feat), dtype=int)
rows = []
for rf, jf, tr, te in iter_block_splits(feat["resume_fold"], feat["job_fold"]):
    assert not set(feat.loc[tr, "resume_id"]) & set(feat.loc[te, "resume_id"]), "Shared resume"
    assert not set(feat.loc[tr, "job_id"]) & set(feat.loc[te, "job_id"]), "Shared job"
    seen[te] += 1
    rows.append({"n_train": len(tr), "n_test": len(te),
                 "n_test_jobs": feat.loc[te, "job_id"].nunique(),
                 "test_with_skills": feat.loc[te, "skills_soft_coverage"].notna().mean()})

print("Each row appears in the test set exactly once :", bool((seen == 1).all()))
print(pd.DataFrame(rows).describe().loc[["min", "mean", "max"]].round(2))

Each row appears in the test set exactly once : True
      n_train  n_test  n_test_jobs  test_with_skills
min   5981.00  336.00          5.0              0.78
mean  6108.16  381.76          5.6              0.81
max   6297.00  414.00          6.0              0.83


In [7]:
feat.to_parquet("../data/processed/v2_features.parquet")
print(feat[["resume_id", "job_id", "resume_fold", "job_fold"]].head())

   resume_id  job_id  resume_fold  job_fold
0          0       0            2         4
1          1       1            3         2
2          2       2            2         2
3          3       3            1         3
4          4       4            2         1
